# Bug Prediction System — V4 (Feature-Enriched)

## Improvements over V3:
- **Removed `confidence` leak** — was correlated -0.98 with target
- **Commit message features** — fix/refactor/wip keyword signals
- **Temporal author features** — rolling bug rate (30d/90d) with `.shift(1)` to prevent leakage
- **Interaction features** — night × complexity, weekend × lines, size × complexity
- **3-way split** — separate calibration set so calibrator never sees training data
- **Optimal threshold** — tuned on calibration set PR curve, not hardcoded 0.5
- **XGBoost** with `scale_pos_weight` for class imbalance

## Dataset:
- 16,722 commits (Python + TypeScript), 873 bugs (5.22%)


## Step 1 — Install & Import Libraries

In [ ]:
!pip install shap xgboost scikit-learn pandas numpy matplotlib seaborn joblib -q
print('Dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, re, warnings
import shap
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
)
warnings.filterwarnings('ignore')
print('All imports successful')

## Step 2 — Load & Inspect Data

In [ ]:
df = pd.read_csv('./combined_output.csv')

# Ensure timestamp column is parsed if present
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
elif 'commit_date' in df.columns:
    df['timestamp'] = pd.to_datetime(df['commit_date'])
    df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nBug distribution:')
print(df['is_buggy'].value_counts())
print(f'Bug rate: {df["is_buggy"].sum() / len(df) * 100:.2f}%')

## Step 3 — Feature Engineering

In [ ]:
# ── 3a. Drop the leaked feature ───────────────────────────────────────────
df_fe = df.drop(columns=['confidence'], errors='ignore').copy()

# ── 3b. Commit message features ───────────────────────────────────────────
if 'commit_message' in df_fe.columns:
    msg = df_fe['commit_message'].fillna('').str.lower()
    df_fe['msg_len']        = msg.str.len()
    df_fe['msg_word_count'] = msg.str.split().str.len()
    df_fe['msg_has_fix']    = msg.str.contains(
        r'\b(fix|bug|error|crash|issue|patch|hotfix|revert)\b', regex=True).astype(int)
    df_fe['msg_has_refactor'] = msg.str.contains(
        r'\b(refactor|clean|rename|move|restructure|simplif)\b', regex=True).astype(int)
    df_fe['msg_is_wip']     = msg.str.contains(
        r'\b(wip|temp|todo|hack|workaround|dirty|quick)\b', regex=True).astype(int)
    df_fe['msg_has_test']   = msg.str.contains(
        r'\b(test|spec|coverage|assert|unittest)\b', regex=True).astype(int)
    df_fe = df_fe.drop(columns=['commit_message'])
    print('Commit message features added')
else:
    print('WARNING: commit_message column not found — skipping message features')
    print('  To add them, ensure your combined_output.csv includes the commit message.')

# ── 3c. Temporal author-rolling features (shift to prevent leakage) ────────
# Sort by time so rolling windows look only backward
if 'timestamp' in df_fe.columns and 'author' in df_fe.columns:
    df_fe = df_fe.sort_values('timestamp').reset_index(drop=True)
    df_fe = df_fe.set_index('timestamp')

    def rolling_bug_rate(group, window):
        shifted = group['is_buggy'].shift(1)   # shift(1) = never see own label
        bugs = shifted.rolling(window, min_periods=1).sum()
        count = shifted.rolling(window, min_periods=1).count()
        return bugs / (count + 1)

    for window, label in [('30D', '30d'), ('90D', '90d')]:
        rates = (df_fe.groupby('author')
                      .apply(lambda g: rolling_bug_rate(g, window))
                      .reset_index(level=0, drop=True))
        df_fe[f'author_bug_rate_{label}'] = rates

    # Days since author's last commit (context-switch risk)
    df_fe['days_since_last_commit'] = (
        df_fe.groupby('author').apply(
            lambda g: g.index.to_series().diff().dt.days.fillna(0)
        ).reset_index(level=0, drop=True)
    )
    df_fe = df_fe.reset_index()
    df_fe[['author_bug_rate_30d','author_bug_rate_90d','days_since_last_commit']] = (
        df_fe[['author_bug_rate_30d','author_bug_rate_90d','days_since_last_commit']].fillna(0)
    )
    print('Temporal author features added')
else:
    print('WARNING: timestamp or author column not found — skipping temporal features')
    print('  Add author + timestamp columns to combined_output.csv for these features.')

# ── 3d. Interaction features ───────────────────────────────────────────────
if 'is_night_commit' in df_fe.columns and 'avg_complexity' in df_fe.columns:
    df_fe['night_x_complexity']  = df_fe['is_night_commit'] * df_fe['avg_complexity']
if 'is_weekend' in df_fe.columns and 'lines_added' in df_fe.columns:
    df_fe['weekend_x_lines']     = df_fe['is_weekend'] * df_fe['lines_added']
if 'avg_complexity' in df_fe.columns and 'lines_added' in df_fe.columns:
    df_fe['size_x_complexity']   = df_fe['lines_added'] * df_fe['avg_complexity']
if 'files_changed' in df_fe.columns and 'avg_complexity' in df_fe.columns:
    df_fe['files_x_complexity']  = df_fe['files_changed'] * df_fe['avg_complexity']
print('Interaction features added')

# ── 3e. Drop non-feature columns ──────────────────────────────────────────
drop_cols = ['timestamp', 'author', 'commit_hash', 'file_path',
             'commit_date', 'repo', 'is_buggy']
drop_cols = [c for c in drop_cols if c in df_fe.columns]

X = df_fe.drop(columns=drop_cols)
y = df_fe['is_buggy']

print(f'\nFinal feature count: {X.shape[1]}')
print(f'Features: {list(X.columns)}')

## Step 4 — Three-Way Split & Encoding

In [ ]:
# Three-way split: train / calibration / test
# Calibration set is held out so the calibrator never sees training data
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp
)

print(f'Train:       {X_train.shape[0]:>5} rows  (bug rate {y_train.mean()*100:.2f}%)')
print(f'Calibration: {X_cal.shape[0]:>5} rows  (bug rate {y_cal.mean()*100:.2f}%)')
print(f'Test:        {X_test.shape[0]:>5} rows  (bug rate {y_test.mean()*100:.2f}%)')

# Encode categorical features — fit on train only
categorical_cols = X_train.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f'\nCategorical columns to encode: {categorical_cols}')

encoders = {}
X_train_enc = X_train.copy()
X_cal_enc   = X_cal.copy()
X_test_enc  = X_test.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train[col].astype(str))
    X_cal_enc[col]   = le.transform(X_cal[col].astype(str))
    X_test_enc[col]  = le.transform(X_test[col].astype(str))
    encoders[col] = le

FEATURE_NAMES = list(X_train_enc.columns)
print(f'Encoding complete. Final feature count: {len(FEATURE_NAMES)}')

## Step 5 — Train XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}  '
      f'(weights bug class {scale_pos_weight:.1f}x heavier)')

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',
    early_stopping_rounds=40,
)

xgb.fit(
    X_train_enc, y_train,
    eval_set=[(X_cal_enc, y_cal)],
    verbose=False,
)
print(f'Best iteration: {xgb.best_iteration}')
print('Training complete')

# Cross-validation AUC on training fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    XGBClassifier(
        n_estimators=xgb.best_iteration or 300,
        max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7,
        scale_pos_weight=scale_pos_weight,
        random_state=42, n_jobs=-1,
    ),
    X_train_enc, y_train, cv=skf, scoring='roc_auc'
)
print(f'CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## Step 6 — Calibration & Threshold Optimisation

In [ ]:
# Calibrate on held-out calibration set (model never saw this data)
calibrated = CalibratedClassifierCV(xgb, method='isotonic', cv='prefit')
calibrated.fit(X_cal_enc, y_cal)

# Find threshold that maximises F1 on the bug class using the calibration set
cal_proba = calibrated.predict_proba(X_cal_enc)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_cal, cal_proba)
f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx    = np.argmax(f1s)
best_thresh = thresholds[best_idx]

print(f'Optimal threshold: {best_thresh:.3f}')
print(f'  At this threshold on calibration set:')
print(f'  Precision={precisions[best_idx]:.3f}  Recall={recalls[best_idx]:.3f}  F1={f1s[best_idx]:.3f}')

## Step 7 — Evaluation on Test Set

In [ ]:
y_proba = calibrated.predict_proba(X_test_enc)[:, 1]
y_pred_opt     = (y_proba >= best_thresh).astype(int)
y_pred_default = (y_proba >= 0.50).astype(int)

auc_roc = roc_auc_score(y_test, y_proba)
pr_auc  = average_precision_score(y_test, y_proba)

print('=' * 56)
print('EVALUATION ON HELD-OUT TEST SET')
print('=' * 56)
print(f'AUC-ROC : {auc_roc:.4f}')
print(f'PR-AUC  : {pr_auc:.4f}   (key metric for imbalanced data)')
print()
print(f'--- Default threshold (0.50) ---')
print(f'Precision: {precision_score(y_test, y_pred_default):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_default):.4f}')
print(f'F1:        {f1_score(y_test, y_pred_default):.4f}')
print()
print(f'--- Optimal threshold ({best_thresh:.3f}) ---')
print(f'Precision: {precision_score(y_test, y_pred_opt):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_opt):.4f}')
print(f'F1:        {f1_score(y_test, y_pred_opt):.4f}')
print()
print(classification_report(y_test, y_pred_opt, target_names=['Safe', 'Bug']))

## Step 8 — Feature Importance

In [ ]:
# XGBoost built-in importance
fi = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'gain':    list(xgb.get_booster().get_score(importance_type='gain').get(f, 0)
                    for f in FEATURE_NAMES),
}).sort_values('gain', ascending=False)

print('TOP 20 FEATURES (XGBoost gain):')
print(fi.head(20).to_string(index=False))

plt.figure(figsize=(10, 7))
sns.barplot(data=fi.head(20), x='gain', y='feature', palette='viridis')
plt.title('Top 20 Features — XGBoost Gain')
plt.xlabel('Mean Gain')
plt.tight_layout()
plt.savefig('feature_importance_v4.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — SHAP Explainability

In [ ]:
print('Computing SHAP values...')
X_sample = X_test_enc.sample(min(500, len(X_test_enc)), random_state=42)
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (mean |SHAP|)')
plt.tight_layout()
plt.savefig('shap_importance_v4.png', dpi=150, bbox_inches='tight')
plt.show()

shap.summary_plot(shap_values, X_sample, show=False)
plt.title('SHAP Beeswarm — Feature Impact on Bug Probability')
plt.tight_layout()
plt.savefig('shap_beeswarm_v4.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP plots saved')

## Step 10 — Permutation Importance

In [ ]:
print('Computing permutation importance...')
perm = permutation_importance(
    calibrated, X_test_enc, y_test,
    n_repeats=10, random_state=42, n_jobs=-1, scoring='roc_auc'
)
perm_df = pd.DataFrame({
    'feature':    FEATURE_NAMES,
    'importance': perm.importances_mean,
    'std':        perm.importances_std,
}).sort_values('importance', ascending=False)

print('TOP 20 FEATURES (Permutation Importance — AUC drop):')
print(perm_df.head(20).to_string(index=False))

plt.figure(figsize=(10, 7))
sns.barplot(data=perm_df.head(20), x='importance', y='feature', palette='coolwarm')
plt.title('Top 20 Features — Permutation Importance (AUC drop)')
plt.xlabel('Mean AUC drop when feature shuffled')
plt.tight_layout()
plt.savefig('permutation_importance_v4.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — Calibration Curve

In [ ]:
raw_proba = xgb.predict_proba(X_test_enc)[:, 1]
cal_proba_test = calibrated.predict_proba(X_test_enc)[:, 1]

frac_pos_raw, mean_pred_raw = calibration_curve(y_test, raw_proba, n_bins=10)
frac_pos_cal, mean_pred_cal = calibration_curve(y_test, cal_proba_test, n_bins=10)

plt.figure(figsize=(7, 5))
plt.plot([0,1],[0,1],'k--', label='Perfect')
plt.plot(mean_pred_raw, frac_pos_raw, 's-', label='XGBoost (uncalibrated)')
plt.plot(mean_pred_cal, frac_pos_cal, 's-', label='Calibrated (isotonic)')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_curve_v4.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12 — ROC & Precision-Recall Curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, 'b-', label=f'AUC = {auc_roc:.3f}')
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec_curve, prec_curve, 'r-', label=f'PR-AUC = {pr_auc:.3f}')
axes[1].axhline(y_test.mean(), color='k', linestyle='--', label='Random baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves_v4.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 13 — Save Model Artifacts

In [ ]:
output_dir = './server/modalv1_v4'
os.makedirs(output_dir, exist_ok=True)

joblib.dump(xgb,          f'{output_dir}/bug_prediction_model.pkl')
joblib.dump(calibrated,   f'{output_dir}/bug_prediction_model_calibrated.pkl')
joblib.dump(encoders,     f'{output_dir}/encoders.pkl')
joblib.dump(FEATURE_NAMES,f'{output_dir}/feature_cols.pkl')

# Persist threshold + metadata so API doesn't hardcode 0.5
model_config = {
    'optimal_threshold': float(best_thresh),
    'feature_names': FEATURE_NAMES,
    'model_type': 'xgboost_isotonic_calibrated',
    'scale_pos_weight': float(scale_pos_weight),
    'auc_roc': float(auc_roc),
    'pr_auc': float(pr_auc),
}
with open(f'{output_dir}/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print(f'Artifacts saved to {output_dir}:')
for fname in os.listdir(output_dir):
    print(f'  {fname}')

## Step 14 — Export Feature Importance & Risk Config

In [ ]:
# Combined importance table
mean_abs_shap = np.abs(shap_values).mean(0)
importance_df = pd.DataFrame({
    'feature':                FEATURE_NAMES,
    'mean_abs_shap':          mean_abs_shap,
    'xgb_gain':               [xgb.get_booster().get_score(importance_type='gain').get(f, 0)
                                for f in FEATURE_NAMES],
    'permutation_importance': perm.importances_mean,
}).sort_values('mean_abs_shap', ascending=False)

print('COMBINED FEATURE IMPORTANCE:')
print(importance_df.head(20).to_string(index=False))
importance_df.to_csv(f'{output_dir}/feature_importance.csv', index=False)

# Risk factor percentile config
risk_config = {}
for feature in FEATURE_NAMES:
    col_data = X_test_enc[feature].values
    risk_config[feature] = {
        'p50': float(np.percentile(col_data, 50)),
        'p75': float(np.percentile(col_data, 75)),
        'p90': float(np.percentile(col_data, 90)),
        'mean': float(col_data.mean()),
    }
with open(f'{output_dir}/risk_factor_config.json', 'w') as f:
    json.dump(risk_config, f, indent=2)

print(f'\nFeature importance + risk config saved to {output_dir}')

## Step 15 — Summary

In [ ]:
print('=' * 56)
print('BUG PREDICTION V4 — COMPLETE')
print('=' * 56)
print(f'  AUC-ROC:          {auc_roc:.4f}')
print(f'  PR-AUC:           {pr_auc:.4f}')
print(f'  Optimal threshold:{best_thresh:.3f}')
print(f'  Precision (bug):  {precision_score(y_test, y_pred_opt):.4f}')
print(f'  Recall (bug):     {recall_score(y_test, y_pred_opt):.4f}')
print(f'  F1 (bug):         {f1_score(y_test, y_pred_opt):.4f}')
print()
print('Artifacts:')
print(f'  {output_dir}/bug_prediction_model_calibrated.pkl')
print(f'  {output_dir}/model_config.json   ← API should read threshold from here')
print(f'  {output_dir}/feature_cols.pkl')
print(f'  {output_dir}/encoders.pkl')
print(f'  {output_dir}/feature_importance.csv')
print()
print('Next steps:')
print('  1. Add commit_message + author + timestamp to combined_output.csv')
print('     to unlock message-signal and temporal-author features')
print('  2. Update API to load threshold from model_config.json')
print('  3. Update get_risk_factors() using feature_importance.csv')
